# 项目一：个人日程规划助手 — Prompt Engineering 入门

在本项目中，你将学习：
1. 如何调用 DeepSeek 大模型 API
2. 理解 API 调用中各参数的含义
3. 使用 System Prompt 约束大模型输出结构化内容
4. 完成一个能将杂乱输入转为清晰时间表的日程规划助手

---

## 一、认识 DeepSeek API

DeepSeek 是一个国产大语言模型，提供了兼容 OpenAI 格式的 API 接口。

### 1.1 获取 API Key

1. 访问 [DeepSeek 开放平台](https://platform.deepseek.com/)
2. 注册账号并登录
3. 在 **API Keys** 页面创建一个新的密钥
4. 妥善保存你的 API Key（不要泄露给他人！）

### 1.2 安装依赖

DeepSeek API 兼容 OpenAI 的 SDK，所以我们只需要安装 `openai` 这个包即可。

In [ ]:
# Install the OpenAI SDK for JupyterLite/Pyodide.
# Shell-style pip commands are not available in this browser runtime.
import micropip
await micropip.install("openai")
print("OpenAI SDK installed")


---
## 二、第一次调用 DeepSeek API

下面是一个**最简调用示例**，让我们看看需要哪些必要参数：

In [ ]:
from openai import OpenAI

# ========== 第一步：创建客户端 ==========
client = OpenAI(
    api_key="sk-xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx",  # 替换为你自己的 API Key
    base_url="https://api.deepseek.com",     # DeepSeek 的 API 地址
)

# ========== 第二步：发送请求 ==========
response = client.chat.completions.create(
    model="deepseek-chat",          # 使用的模型名称
    messages=[
        {"role": "user", "content": "你好，请简单介绍一下你自己"}
    ]
)

# ========== 第三步：解析返回结果 ==========
print(response.choices[0].message.content)

---
## 三、API 参数详解

下面逐一讲解 `chat.completions.create()` 中各个参数的作用：

### 3.1 `api_key` — 身份凭证

| 属性 | 说明 |
|------|------|
| 类型 | `string` |
| 必填 | 是 |
| 说明 | 你的 API 密钥，用于身份认证和计费。格式通常为 `sk-` 开头 |

> ⚠️ **安全提示**：永远不要把 API Key 硬编码在代码里提交到 GitHub！建议使用环境变量：
> ```python
> import os
> api_key = os.getenv("DEEPSEEK_API_KEY")
> ```

### 3.2 `base_url` — API 服务地址

| 属性 | 说明 |
|------|------|
| 类型 | `string` |
| 必填 | 是（非 OpenAI 官方服务时） |
| DeepSeek 地址 | `https://api.deepseek.com` |
| 说明 | 指定请求发送到哪个服务器。因为 DeepSeek 兼容 OpenAI 格式，所以通过修改这个地址来切换不同的模型服务商 |

### 3.3 `model` — 模型选择

| 属性 | 说明 |
|------|------|
| 类型 | `string` |
| 必填 | 是 |
| 可选值 | `deepseek-chat`（通用对话）、`deepseek-reasoner`（深度推理） |
| 说明 | 决定使用哪个模型。`deepseek-chat` 适合日常对话和大多数任务，`deepseek-reasoner` 适合复杂推理 |

### 3.4 `messages` — 消息列表（核心参数）

| 属性 | 说明 |
|------|------|
| 类型 | `list[dict]` |
| 必填 | 是 |
| 说明 | 对话的消息列表，每条消息包含 `role` 和 `content` |

消息有三种角色（role）：

| role | 含义 | 用途 |
|------|------|------|
| `system` | 系统消息 | 设定 AI 的行为规则、角色和输出格式 |
| `user` | 用户消息 | 用户输入的问题或指令 |
| `assistant` | 助手消息 | AI 之前的回复，用于多轮对话上下文 |

```python
messages=[
    {"role": "system", "content": "你是一个专业的翻译助手，只输出翻译结果。"},
    {"role": "user",   "content": "请把'你好世界'翻译成英文"}
]
```

### 3.5 `temperature` — 创造性控制

| 属性 | 说明 |
|------|------|
| 类型 | `float`，范围 0.0 ~ 2.0 |
| 默认值 | 1.0 |
| 说明 | 控制输出的随机性。值越低输出越确定、越保守；值越高输出越多样、越有创意 |

| 场景 | 推荐值 |
|------|--------|
| 日程规划、数据提取（需要稳定输出） | 0.0 ~ 0.3 |
| 一般对话 | 0.7 ~ 1.0 |
| 创意写作、头脑风暴 | 1.2 ~ 1.5 |

### 3.6 `max_tokens` — 输出长度限制

| 属性 | 说明 |
|------|------|
| 类型 | `int` |
| 说明 | 限制模型最多生成多少个 token（1 个中文字约 1~2 个 token）。设置过小会导致回复被截断 |

### 3.7 完整参数示例

下面是一个包含所有常用参数的完整调用：

In [ ]:
from openai import OpenAI

client = OpenAI(
    api_key="sk-xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx",  # 替换为你的 API Key
    base_url="https://api.deepseek.com",
)

response = client.chat.completions.create(
    model="deepseek-chat",          # 模型选择
    messages=[
        {"role": "system", "content": "你是一个专业的日程规划助手。"},  # 系统设定
        {"role": "user",   "content": "我是一位胸有大志的研0新生，帮我安排明天的日程"},           # 用户输入
    ],
    temperature=0.2,               # 低温度 → 输出稳定、格式可控
    max_tokens=1000,                # 最多生成 1000 个 token
)

# 获取回复内容
result = response.choices[0].message.content
print(result)

# 查看 token 使用情况
print(f"\n--- Token 用量 ---")
print(f"输入 tokens: {response.usage.prompt_tokens}")
print(f"输出 tokens: {response.usage.completion_tokens}")
print(f"总计 tokens: {response.usage.total_tokens}")

---
## 四、封装一个可复用的调用函数

在实际项目中，我们通常会把 API 调用封装成函数，方便反复使用：

In [ ]:
import os
from openai import OpenAI

# 推荐使用环境变量存储 API Key，避免泄露
# 在终端中设置：set DEEPSEEK_API_KEY=sk-xxxxxxxx
client = OpenAI(
    api_key=os.getenv("DEEPSEEK_API_KEY", "sk-xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx"),  # 优先读环境变量，否则用默认值
    base_url="https://api.deepseek.com",
)

def call_deepseek(system_prompt, user_input, temperature=0.2, max_tokens=500):
    """
    调用 DeepSeek API 的通用函数
    
    参数:
        system_prompt: str  - 系统提示词，定义 AI 的角色和行为
        user_input: str     - 用户输入的内容
        temperature: float  - 创造性控制 (0.0~2.0)，默认 0.2
        max_tokens: int     - 最大输出 token 数，默认 500
    返回:
        str - 模型的回复内容
    """
    response = client.chat.completions.create(
        model="deepseek-chat",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_input},
        ],
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return response.choices[0].message.content


# 测试一下
result = call_deepseek(
    system_prompt="你是一个友好的助手，用一句话回答问题。",
    user_input="简短的介绍一下河南大学？"
)
print(result)

---
## 五、实战：构建日程规划助手

### 核心思路

用户说的话往往是**杂乱无章**的，比如：
> "明天上午九点有个会，记得11点前要把报告写完，下午两点得集中回一下邮件。"

我们的目标是让 AI 把它整理成**结构化的时间表**：
> - 09:00 开会  
> - 11:00 写报告  
> - 14:00 回复邮件  

**关键就在于 System Prompt 的设计**——它决定了 AI 如何理解和格式化输出。

### 5.1 补全下面的代码

In [ ]:
def generate_schedule(user_input):
    """
    日程规划助手：将用户杂乱的输入转化为清晰的 Markdown 时间表
    """
    # ========== 你的代码开始 ==========
    
    # 第一步：编写 System Prompt
    # 要求：
    #   1. 定义 AI 的角色（日程规划助手）
    #   2. 规定输出格式为 Markdown 列表
    #   3. 时间统一使用 24 小时制的 HH:MM 格式
    #   4. 不要输出多余的解释，只输出时间表
    system_prompt = """TODO"""
    
    # 第二步：调用 call_deepseek 函数
    response = call_deepseek(system_prompt, user_input)
    
    # ========== 你的代码结束 ==========
    return response


# 测试
user_notes = "明天上午九点有个会，记得11点前要把报告写完，下午两点得集中回一下邮件。"
print(generate_schedule(user_notes))

---
## 六、课后练习

### 练习 1：基础巩固 — 翻译助手

编写一个 `translate_text` 函数，实现中英文互译。

**要求：**
- System Prompt 中定义 AI 为翻译助手
- 用户输入中文时翻译成英文，输入英文时翻译成中文
- 只输出翻译结果，不要附加解释

**提示：** 在 System Prompt 中加入判断逻辑的指引

In [ ]:
# ===== 练习 1：在这里编写你的代码 =====

def translate_text(text):
    # TODO: 补全 System Prompt 和函数逻辑
    system_prompt = """TODO"""
    response = call_deepseek(system_prompt, text)
    return response

# 测试用例
print(translate_text("人工智能正在改变世界"))   # 期望输出：英文翻译
print(translate_text("Artificial intelligence is changing the world"))  # 期望输出：中文翻译

### 练习 2：进阶挑战 — 情绪分析器

编写一个 `analyze_sentiment` 函数，分析用户输入文本的情绪。

**要求：**
- 输出格式为 JSON：`{"emotion": "xxx", "confidence": "高/中/低", "summary": "一句话总结"}`
- 支持的情绪包括：开心、悲伤、愤怒、焦虑、平静、其他
- System Prompt 中要明确输出格式要求

**思考：** `temperature` 应该设置为多少？为什么？

In [ ]:
# ===== 练习 2：在这里编写你的代码 =====

def analyze_sentiment(text):
    # TODO: 补全 System Prompt 和函数逻辑
    # 提示：temperature 建议设低，因为需要稳定的 JSON 格式输出
    system_prompt = """TODO"""
    response = call_deepseek(system_prompt, text, temperature=0.2)


    return response

# 测试用例
print(analyze_sentiment("太棒了！我今天拿到了心仪已久的 offer！"))
print(analyze_sentiment("哎，又下雨了，出门真不方便..."))
print(analyze_sentiment("这个 bug 改了三遍还是不对！！！"))
print(analyze_sentiment("我考上河南大学了！！！"))

### 练习 3：拓展思考

回答以下问题（可以写在代码注释里，也可以口头讨论）：

1. **System Prompt 和 User Prompt 的区别是什么？** 为什么不把所有指令都放在 User Prompt 里？

2. **为什么日程规划助手的 `temperature` 要设得比较低？** 如果设成 1.5 会发生什么？

3. **如果要实现多轮对话**（比如用户说"把下午的会议改到三点"），`messages` 列表应该怎么构造？试着写出来。

In [ ]:
# ===== 练习 3：在这里写下你的思考 =====

# 问题 1：
# 你的回答：

# 问题 2：
# 你的回答：

# 问题 3：尝试写出多轮对话的 messages 列表
# messages = [
#     ...
# ]

---
## 小结

恭喜你完成了第一个项目！回顾一下你学到的知识：

| 知识点 | 要点 |
|--------|------|
| API 调用 | 使用 `openai` SDK，设置 `base_url` 指向 DeepSeek |
| `api_key` | 身份凭证，务必保密，推荐用环境变量 |
| `model` | 选择模型，`deepseek-chat` 适合大多数场景 |
| `messages` | 包含 system / user / assistant 三种角色的消息列表 |
| `system` prompt | 定义 AI 的角色、行为和输出格式，是 Prompt Engineering 的核心 |
| `temperature` | 控制随机性，需要稳定输出时设低，需要创意时设高 |
| `max_tokens` | 限制输出长度，防止截断或浪费 |

**下一步预告：** 在[项目二](./project2_tool_use.ipynb)中，我们将学习如何让智能体调用外部工具（Tool Use），让它不仅能"想"，还能"做"！